In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

n = 200

df = pd.DataFrame({
    'ticket_id': range(1, n + 1),
    'venta': np.round(
        np.concatenate([
            np.random.normal(loc=350, scale=80, size=180),  # ventas normales
            np.random.normal(loc=1800, scale=200, size=20)  # outliers: mesas grandes
        ]), 2
    ),
    'num_productos': np.random.randint(1, 12, size=n),
    'mesa': np.random.choice(['terraza', 'interior', 'barra'], size=n, p=[0.4, 0.45, 0.15])
})

# Forzar que no haya ventas negativas
df['venta'] = df['venta'].clip(lower=50)

print(df.shape)
print(df.head(10))
print(df.describe())


(200, 4)
   ticket_id   venta  num_productos      mesa
0          1  389.74              2   terraza
1          2  338.94             10  interior
2          3  401.82              1  interior
3          4  471.84              8   terraza
4          5  331.27              1     barra
5          6  331.27              9  interior
6          7  476.34             11   terraza
7          8  411.39              6     barra
8          9  312.44              7     barra
9         10  393.40             10     barra
        ticket_id        venta  num_productos
count  200.000000   200.000000     200.000000
mean   100.500000   489.237100       5.780000
std     57.879185   432.561185       3.154577
min      1.000000   140.420000       1.000000
25%     50.750000   304.940000       3.000000
50%    100.500000   361.290000       6.000000
75%    150.250000   415.770000       9.000000
max    200.000000  1971.280000      11.000000


In [2]:
# ============================================================
# S4.1 — MEDIA Y MEDIANA
# ============================================================

ventas = df['venta']

# --- MEDIA ---
# Fórmula: suma de todos los valores / número de observaciones
# x̄ = (Σ xᵢ) / n

media_manual = ventas.sum() / len(ventas)
media_numpy  = np.mean(ventas)
media_pandas = ventas.mean()

print("=== MEDIA ===")
print(f"  Manual:  {media_manual:.2f}")
print(f"  NumPy:   {media_numpy:.2f}")
print(f"  Pandas:  {media_pandas:.2f}")

# --- MEDIANA ---
# Fórmula: valor central cuando los datos están ordenados
# Si n es par: promedio de los dos valores centrales

mediana_numpy  = np.median(ventas)
mediana_pandas = ventas.median()

print("\n=== MEDIANA ===")
print(f"  NumPy:   {mediana_numpy:.2f}")
print(f"  Pandas:  {mediana_pandas:.2f}")

# --- COMPARACIÓN ---
diferencia = media_pandas - mediana_pandas
print(f"\n=== DIFERENCIA media - mediana ===")
print(f"  {media_pandas:.2f} - {mediana_pandas:.2f} = {diferencia:.2f}")
print(f"  {'⚠️  Hay outliers jalando el promedio hacia arriba' if diferencia > 50 else '✅ Distribución relativamente simétrica'}")

=== MEDIA ===
  Manual:  489.24
  NumPy:   489.24
  Pandas:  489.24

=== MEDIANA ===
  NumPy:   361.29
  Pandas:  361.29

=== DIFERENCIA media - mediana ===
  489.24 - 361.29 = 127.95
  ⚠️  Hay outliers jalando el promedio hacia arriba


In [3]:
# ============================================================
# S4.1 — VARIANZA Y DESVIACIÓN ESTÁNDAR
# ============================================================

# --- VARIANZA ---
# Fórmula: promedio de las distancias al cuadrado respecto a la media
# σ² = Σ(xᵢ - x̄)² / n        ← poblacional
# s² = Σ(xᵢ - x̄)² / (n - 1)  ← muestral (la más común en DS)

varianza_manual = ((ventas - ventas.mean()) ** 2).sum() / (len(ventas) - 1)
varianza_numpy  = np.var(ventas, ddof=1)   # ddof=1 → muestral
varianza_pandas = ventas.var()              # pandas usa muestral por default

print("=== VARIANZA (muestral) ===")
print(f"  Manual:  {varianza_manual:.2f}")
print(f"  NumPy:   {varianza_numpy:.2f}")
print(f"  Pandas:  {varianza_pandas:.2f}")

# --- DESVIACIÓN ESTÁNDAR ---
# Fórmula: raíz cuadrada de la varianza
# s = √s²  ← regresa a las unidades originales (pesos, en este caso)

std_manual = varianza_manual ** 0.5
std_numpy  = np.std(ventas, ddof=1)
std_pandas = ventas.std()

print("\n=== DESVIACIÓN ESTÁNDAR ===")
print(f"  Manual:  {std_manual:.2f}")
print(f"  NumPy:   {std_numpy:.2f}")
print(f"  Pandas:  {std_pandas:.2f}")

# --- INTERPRETACIÓN ---
print(f"\n=== INTERPRETACIÓN ===")
print(f"  Media:   ${ventas.mean():.2f}")
print(f"  ± std:   ${ventas.std():.2f}")
print(f"  Rango típico: ${ventas.mean() - ventas.std():.2f} a ${ventas.mean() + ventas.std():.2f}")

=== VARIANZA (muestral) ===
  Manual:  187109.18
  NumPy:   187109.18
  Pandas:  187109.18

=== DESVIACIÓN ESTÁNDAR ===
  Manual:  432.56
  NumPy:   432.56
  Pandas:  432.56

=== INTERPRETACIÓN ===
  Media:   $489.24
  ± std:   $432.56
  Rango típico: $56.68 a $921.80


In [4]:
# ============================================================
# S4.1 — PERCENTILES, CUARTILES E IQR
# ============================================================

# --- PERCENTILES ---
# El percentil P indica que el P% de los datos están por debajo de ese valor
# Ejemplo: percentil 90 = el 90% de los tickets valen menos que ese número

p10 = np.percentile(ventas, 10)
p25 = np.percentile(ventas, 25)   # Q1
p50 = np.percentile(ventas, 50)   # Q2 = mediana
p75 = np.percentile(ventas, 75)   # Q3
p90 = np.percentile(ventas, 90)
p95 = np.percentile(ventas, 95)

print("=== PERCENTILES ===")
print(f"  P10:  ${p10:.2f}  → 10% de tickets valen menos que esto")
print(f"  P25:  ${p25:.2f}  → Q1")
print(f"  P50:  ${p50:.2f}  → mediana (Q2)")
print(f"  P75:  ${p75:.2f}  → Q3")
print(f"  P90:  ${p90:.2f}  → 90% de tickets valen menos que esto")
print(f"  P95:  ${p95:.2f}  → solo el 5% supera este valor")

# --- IQR ---
# Rango intercuartílico: distancia entre Q3 y Q1
# Contiene el 50% central de los datos
# Fórmula: IQR = Q3 - Q1

iqr = p75 - p25

print(f"\n=== IQR (Rango Intercuartílico) ===")
print(f"  Q1:   ${p25:.2f}")
print(f"  Q3:   ${p75:.2f}")
print(f"  IQR:  ${iqr:.2f}  → el 50% central de tickets cae en este rango")

# --- DETECCIÓN DE OUTLIERS CON IQR ---
# Regla estándar: outlier si está fuera de [Q1 - 1.5*IQR, Q3 + 1.5*IQR]

limite_inf = p25 - 1.5 * iqr
limite_sup = p75 + 1.5 * iqr

outliers = ventas[(ventas < limite_inf) | (ventas > limite_sup)]

print(f"\n=== DETECCIÓN DE OUTLIERS ===")
print(f"  Límite inferior: ${limite_inf:.2f}")
print(f"  Límite superior: ${limite_sup:.2f}")
print(f"  Outliers encontrados: {len(outliers)}")
print(f"  Valores: {sorted(outliers.values)[:5]}... (primeros 5)")

=== PERCENTILES ===
  P10:  $254.10  → 10% de tickets valen menos que esto
  P25:  $304.94  → Q1
  P50:  $361.29  → mediana (Q2)
  P75:  $415.77  → Q3
  P90:  $660.55  → 90% de tickets valen menos que esto
  P95:  $1786.74  → solo el 5% supera este valor

=== IQR (Rango Intercuartílico) ===
  Q1:   $304.94
  Q3:   $415.77
  IQR:  $110.83  → el 50% central de tickets cae en este rango

=== DETECCIÓN DE OUTLIERS ===
  Límite inferior: $138.70
  Límite superior: $582.01
  Outliers encontrados: 20
  Valores: [1497.03, 1550.85, 1571.41, 1585.82, 1623.23]... (primeros 5)
